# Notebook 08 – Điều gì đi cùng rủi ro cao, **ngoài** các dấu hiệu túng thiếu hiển nhiên?

Trễ hạn, vay nhiều nơi, nợ nặng, đảo nợ là những dấu hiệu túng thiếu mà ai cũng biết. Câu hỏi ở đây: **sau khi đã tính đến các dấu hiệu đó**, còn đặc điểm nào khác của hồ sơ đi cùng với vỡ nợ?

Phương pháp (chỉ dùng cột có sẵn, không tạo biến mới):
1. **Mô hình "hiển nhiên"**: LightGBM chỉ dùng 4 nhóm hành vi hiển nhiên → xác suất vỡ nợ *kỳ vọng* của mỗi hồ sơ.
2. **Giá trị gia tăng**: so sánh với mô hình cuối (thêm mọi đặc trưng còn lại) bằng kiểm định DeLong.
3. **SHAP**: xếp hạng các đặc trưng *không hiển nhiên* theo đóng góp vào dự báo của mô hình cuối, kèm chiều tác động.
4. **Rủi ro vượt kỳ vọng**: với từng đặc trưng không hiển nhiên, so sánh số vỡ nợ **quan sát** với số vỡ nợ **kỳ vọng** từ mô hình hiển nhiên theo từng nhóm giá trị; kiểm định z ( |z| > 3 ⇒ chênh lệch có ý nghĩa thống kê ).
5. **Rủi ro ẩn ở khách "trông an toàn"**: trong nhóm 60% hồ sơ có rủi ro hiển nhiên thấp nhất, cây quyết định nông (độ sâu 3) tìm các phân khúc có tỷ lệ vỡ nợ cao bất thường – ra quy tắc dễ đọc.

**Lưu ý diễn giải**: đây là **tương quan**, không phải nhân quả. Thuộc tính nhạy cảm (giới tính, tuổi…) được đánh dấu riêng: chúng có thể tương quan với rủi ro nhưng không nên dùng để từ chối khách hàng.

In [ ]:
# Giai phong bo nho con lai tu cac phan truoc (chay TRUOC khi doc du lieu cua phan nay)
import utils
utils.free_memory(globals())

In [ ]:
import warnings, re; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, polars as pl, joblib
import matplotlib.pyplot as plt
import utils
from utils import META
import lightgbm as lgb
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_auc_score
from sklearn.tree import DecisionTreeClassifier, export_text

OUT_DIR, SEED = utils.get_out_dir(), 42
pd.set_option("display.max_colwidth", 80)
defs = utils.load_definitions()
pack = joblib.load(OUT_DIR / "final_model.joblib"); final, feats = pack["model"], pack["features"]
df = pl.read_parquet(OUT_DIR / "features_v2.parquet").to_pandas()
grp = pd.read_csv(OUT_DIR / "behavior_groups.csv", index_col=0)["nhom"]
OBVIOUS = {"tre_han", "vay_nhieu_noi", "ganh_nang_no", "dao_no"}
obv = [c for c in feats if grp.get(c) in OBVIOUS]
other = [c for c in feats if c not in obv]
desc = utils.describe(feats, defs)
SENS = re.compile(r"\bgender\b|\bsex\b|birth|\bage\b|marital|religio|nationality", re.I)
sensitive = [c for c in other if SENS.search(desc[c] or "")]
train, valid, test = utils.time_split(df)
print("Dac trung hien nhien:", len(obv), "| khong hien nhien:", len(other), "| nhay cam:", sensitive)

## 1–2. Mô hình chỉ dùng dấu hiệu hiển nhiên và giá trị gia tăng của phần còn lại

In [ ]:
P = dict(n_estimators=3000, learning_rate=0.03, num_leaves=31, min_child_samples=200, subsample=0.8, subsample_freq=1,
         colsample_bytree=0.6, reg_lambda=5.0, random_state=SEED, n_jobs=-1, verbose=-1, metric="auc")
m_obv = lgb.LGBMClassifier(**P).fit(train[obv], train["target"], eval_set=[(valid[obv], valid["target"])],
                                    callbacks=[lgb.early_stopping(200, verbose=False)])
iso = IsotonicRegression(y_min=1e-4, y_max=1 - 1e-4, out_of_bounds="clip").fit(m_obv.predict_proba(valid[obv])[:, 1], valid["target"])
for d in (train, valid, test):
    d["p_obv"] = iso.predict(m_obv.predict_proba(d[obv])[:, 1])      # xac suat vo no ky vong tu dau hieu hien nhien
p_final = final.predict_proba(test[feats])[:, 1]
dl = utils.delong_test(test["target"], p_final, test["p_obv"])
print("AUC test - chi dau hieu hien nhien: %.4f | mo hinh cuoi: %.4f | chenh %.4f (z = %.1f, p = %.2g)" % (
      dl["AUC_2"], dl["AUC_1"], dl["chenh"], dl["z"], dl["p_value"]))
pd.Series({"AUC_hien_nhien": dl["AUC_2"], "AUC_cuoi": dl["AUC_1"], "chenh": dl["chenh"], "p_value": dl["p_value"]}).to_csv(OUT_DIR / "obvious_vs_final.csv")

## 3. Đặc trưng không hiển nhiên đóng góp nhiều nhất (SHAP, mô hình cuối)
`chieu`: **+** nghĩa là giá trị càng cao thì dự báo rủi ro càng cao (tương quan Spearman giữa giá trị và SHAP); với biến phân loại xem bảng ở mục 4.

In [ ]:
import shap
Xs = test.sample(min(len(test), 20_000), random_state=SEED)
sv = shap.TreeExplainer(final.booster_).shap_values(Xs[feats]); sv = sv[1] if isinstance(sv, list) else sv
S = pd.DataFrame(sv, columns=feats, index=Xs.index)
imp = S.abs().mean()
rows = []
for c in other:
    x = Xs[c]
    sign = (pd.Series(pd.to_numeric(x, errors="coerce")).corr(S[c], method="spearman")
            if pd.api.types.is_numeric_dtype(x) else np.nan)
    rows.append({"dac_trung": c, "ty_trong_SHAP_%": 100 * imp[c] / imp.sum(),
                 "chieu": "" if pd.isna(sign) else ("+" if sign > 0 else "-"),
                 "nhay_cam": c in sensitive, "mo_ta": desc[c]})
top_other = pd.DataFrame(rows).sort_values("ty_trong_SHAP_%", ascending=False).set_index("dac_trung")
print("Ty trong SHAP: hien nhien %.1f%% | khong hien nhien %.1f%%" % (100 * imp[obv].sum() / imp.sum(), 100 * imp[other].sum() / imp.sum()))
top_other.to_csv(OUT_DIR / "non_obvious_shap.csv"); top_other.head(20).round(2)

## 4. Rủi ro vượt kỳ vọng: quan sát so với kỳ vọng từ dấu hiệu hiển nhiên (tập test)
Với mỗi nhóm giá trị của một đặc trưng: `O` = số vỡ nợ thực tế, `E` = tổng xác suất kỳ vọng từ mô hình hiển nhiên,
`O/E` > 1 ⇒ nhóm này vỡ nợ **nhiều hơn** mức mà các dấu hiệu túng thiếu giải thích được. `z = (O − E) / √Σp(1−p)`; |z| > 3 xem là có ý nghĩa thống kê (đã tính đến việc kiểm định nhiều nhóm).

In [ ]:
def excess_table(d, col, q=5, min_n=500):
    x = d[col]
    if pd.api.types.is_numeric_dtype(x):
        b = pd.qcut(x, q, duplicates="drop")
        b = b.cat.add_categories("thieu du lieu").fillna("thieu du lieu")       # giu thu tu khoang gia tri
    else:
        top = x.astype(str).value_counts().index[:8]
        b = x.astype(str).where(x.astype(str).isin(top), "khac").where(x.notna(), "thieu du lieu")
    g = d.assign(b=b).groupby("b", observed=True, sort=True).agg(n=("target", "size"), O=("target", "sum"), E=("p_obv", "sum"),
                                      V=("p_obv", lambda p: (p * (1 - p)).sum()))
    g = g[g["n"] >= min_n]
    g["ty_le_vo_no"] = g["O"] / g["n"]; g["O/E"] = g["O"] / g["E"]; g["z"] = (g["O"] - g["E"]) / np.sqrt(g["V"])
    return g.drop(columns="V")

TOP_N = 12
out, summ = {}, []
for c in top_other.index[:TOP_N]:
    t = excess_table(test, c); out[c] = t
    k = t["z"].abs().idxmax()
    summ.append({"dac_trung": c, "nhom_chenh_nhieu_nhat": k, "O/E": t.loc[k, "O/E"], "z": t.loc[k, "z"],
                 "ty_le_vo_no_nhom": t.loc[k, "ty_le_vo_no"], "nhay_cam": c in sensitive, "mo_ta": desc[c]})
summ = pd.DataFrame(summ).set_index("dac_trung")
summ["co_y_nghia"] = summ["z"].abs() > 3
summ.to_csv(OUT_DIR / "non_obvious_excess_risk.csv")
print("Ty le vo no chung (test): %.2f%%" % (100 * test["target"].mean())); display(summ.round(3))

In [ ]:
# Bang chi tiet cho 4 dac trung co chenh lech lon nhat
for c in summ["z"].abs().sort_values(ascending=False).index[:4]:
    print(f"\n=== {c}: {desc[c]}" + ("   [THUOC TINH NHAY CAM]" if c in sensitive else ""))
    display(out[c][["n", "ty_le_vo_no", "O", "E", "O/E", "z"]].round(3))

## 5. Rủi ro ẩn ở khách hàng "trông an toàn"
Nhóm "trông an toàn" = 60% hồ sơ có xác suất kỳ vọng từ dấu hiệu hiển nhiên thấp nhất (ngưỡng lấy từ train).
Cây quyết định độ sâu 3, chỉ dùng đặc trưng **số** không hiển nhiên và **không nhạy cảm**, học trên train (nhóm an toàn) và đánh giá trên test (nhóm an toàn).
Giá trị thiếu được thay bằng −999 để cây tách riêng nhóm thiếu dữ liệu (không suy diễn giá trị).

In [ ]:
thr = np.quantile(train["p_obv"], 0.60)
safe_tr, safe_te = train[train["p_obv"] <= thr], test[test["p_obv"] <= thr]
num_other = [c for c in top_other.index if c not in sensitive and pd.api.types.is_numeric_dtype(df[c])][:30]
tree = DecisionTreeClassifier(max_depth=3, min_samples_leaf=max(500, len(safe_tr) // 200), class_weight=None, random_state=SEED)
tree.fit(safe_tr[num_other].fillna(-999), safe_tr["target"])
leaf_te = tree.apply(safe_te[num_other].fillna(-999))
base_safe = safe_te["target"].mean()
leaves = (safe_te.assign(la=leaf_te).groupby("la").agg(so_ho_so=("target", "size"), ty_le_vo_no=("target", "mean"), pd_ky_vong=("p_obv", "mean")))
leaves["lift_trong_nhom_an_toan"] = leaves["ty_le_vo_no"] / base_safe
leaves = leaves.sort_values("ty_le_vo_no", ascending=False)
print("Nhom trong an toan (test): %d ho so | ty le vo no %.2f%% (chung %.2f%%) | AUC cay tren test: %.3f" % (
      len(safe_te), 100 * base_safe, 100 * test["target"].mean(), roc_auc_score(safe_te["target"], tree.predict_proba(safe_te[num_other].fillna(-999))[:, 1])))
leaves.to_csv(OUT_DIR / "hidden_risk_leaves.csv"); display(leaves.round(4))
print(export_text(tree, feature_names=num_other, show_weights=False, decimals=1))
print("Mo ta cac bien trong cay:"); display(desc[[c for c in num_other if c in export_text(tree, feature_names=num_other)]].rename("mo_ta").to_frame())

## 6. Nhận xét (tự viết)
- Phần rủi ro mà dấu hiệu túng thiếu hiển nhiên **không** giải thích được lớn đến đâu (chênh AUC, tỷ trọng SHAP): ...
- Các đặc điểm không hiển nhiên nổi bật (sản phẩm/kênh, thu nhập – nghề nghiệp, thông tin liên hệ dùng chung, thời gian quan hệ…) và nhóm giá trị có O/E cao, |z| > 3: ...
- Phân khúc rủi ro ẩn trong nhóm "trông an toàn" và cách dùng trong thực tế (xác minh thêm, hạn mức thấp hơn, theo dõi sau giải ngân): ...
- Thuộc tính nhạy cảm: có tương quan không; vì sao không nên dùng để từ chối: ...
- Hạn chế: tương quan không phải nhân quả; tên một số giá trị phân loại đã bị ban tổ chức mã hoá.